# Thiết lập dữ liệu thí nghiệm dùng chung

## Mini Capstone Machine Learning

Notebook này tạo **một lần duy nhất** danh sách mẫu train/test dùng chung cho SVM, KNN và Decision Tree. Mục tiêu là so sánh ba thuật toán trên cùng điều kiện dữ liệu. Notebook không chuẩn hóa dữ liệu và không huấn luyện mô hình.

## 1. Quy ước thí nghiệm đã thống nhất

- Đầu vào: `data/processed/breast_cancer_clean.csv`.
- Nhãn: `diagnosis`, với `B = 0` và `M = 1` chỉ dùng để chia dữ liệu có stratify.
- Tỷ lệ split: 80% train và 20% test.
- `random_state = 42` để có thể tái lập.
- Tập test chỉ dùng ở bước đánh giá cuối của từng mô hình; không dùng để tuning siêu tham số.
- Các thao tác chuẩn hóa hoặc chọn đặc trưng phải làm trong pipeline của từng notebook mô hình, chỉ fit trên phần train.

In [1]:
from pathlib import Path
import hashlib
import json
from datetime import datetime, timezone

import pandas as pd
from sklearn.model_selection import train_test_split

TEST_SIZE = 0.20
RANDOM_STATE = 42

project_root = Path.cwd().resolve()
if project_root.name == 'notebooks':
    project_root = project_root.parent

clean_data_path = project_root / 'data' / 'processed' / 'breast_cancer_clean.csv'
splits_dir = project_root / 'data' / 'splits'
train_indices_path = splits_dir / 'train_indices.csv'
test_indices_path = splits_dir / 'test_indices.csv'
metadata_path = splits_dir / 'split_metadata.json'

if not clean_data_path.exists():
    raise FileNotFoundError(
        f'Không tìm thấy {clean_data_path}. Hãy chạy notebook 01 để tạo dữ liệu sạch trước.'
    )

print(f'Dữ liệu sạch: {clean_data_path}')
print(f'Thư mục lưu split: {splits_dir}')

Dữ liệu sạch: D:\Project_AIL303m\data\processed\breast_cancer_clean.csv
Thư mục lưu split: D:\Project_AIL303m\data\splits


## 2. Đọc và xác minh dữ liệu sạch

Notebook chỉ tạo split khi dữ liệu sạch còn đầy đủ và đúng cấu trúc. `row_index` là số thứ tự dòng trong file sạch hiện tại; các notebook mô hình sau này phải dùng `iloc` với danh sách index này.

In [2]:
clean_data = pd.read_csv(clean_data_path)

if 'diagnosis' not in clean_data.columns:
    raise ValueError('Dữ liệu sạch phải có cột diagnosis.')
if clean_data.isna().any().any():
    raise ValueError('Dữ liệu sạch còn giá trị thiếu. Hãy kiểm tra lại notebook 01.')
if not clean_data['diagnosis'].isin(['B', 'M']).all():
    raise ValueError('Cột diagnosis chỉ được chứa B hoặc M.')

y = clean_data['diagnosis'].map({'B': 0, 'M': 1})
row_indices = clean_data.index.to_numpy()

print(f'Kích thước dữ liệu sạch: {clean_data.shape[0]} hàng x {clean_data.shape[1]} cột')
print('Phân bố nhãn toàn bộ dữ liệu:')
display(clean_data['diagnosis'].value_counts().rename_axis('diagnosis').to_frame('so_mau'))

Kích thước dữ liệu sạch: 569 hàng x 31 cột
Phân bố nhãn toàn bộ dữ liệu:


,so_mau
diagnosis,
B,357
M,212


## 3. Kiểm tra an toàn trước khi tạo split

Split là điều kiện chung của toàn bộ dự án. Nếu ba file split đã tồn tại, notebook sẽ dừng thay vì ghi đè. Chỉ xóa thủ công cả ba file khi nhóm đã thống nhất thay đổi thiết kế thí nghiệm và ghi rõ lý do trong báo cáo.

In [3]:
existing_split_files = [
    path for path in [train_indices_path, test_indices_path, metadata_path] if path.exists()
]

if existing_split_files:
    existing_names = ', '.join(path.name for path in existing_split_files)
    raise FileExistsError(
        f'Split đã tồn tại ({existing_names}). Không ghi đè để bảo vệ tính công bằng. '
        'Nếu nhóm thật sự cần tạo split mới, hãy thống nhất lý do, xóa cả ba file và chạy lại notebook.'
    )

print('Chưa có split cũ. Có thể tạo split mới.')

Chưa có split cũ. Có thể tạo split mới.


## 4. Tạo train/test split cố định

`stratify=y` giữ tỷ lệ lành tính/ác tính gần giống dữ liệu gốc ở cả train và test. Chỉ số dòng được lưu, không lưu bản sao dữ liệu train/test, để mọi mô hình luôn đọc từ cùng một file sạch.

In [4]:
train_indices, test_indices = train_test_split(
    row_indices,
    test_size=0.20,
    stratify=y,
    random_state=42,
)

train_indices = sorted(train_indices)
test_indices = sorted(test_indices)

train_data = clean_data.iloc[train_indices]
test_data = clean_data.iloc[test_indices]

split_summary = pd.DataFrame({
    'toan_bo': clean_data['diagnosis'].value_counts(),
    'train': train_data['diagnosis'].value_counts(),
    'test': test_data['diagnosis'].value_counts(),
}).fillna(0).astype(int)

split_summary.loc['Tong'] = [len(clean_data), len(train_data), len(test_data)]
display(split_summary)

assert len(train_data) + len(test_data) == len(clean_data)
assert set(train_indices).isdisjoint(test_indices)
assert len(train_indices) == 455 and len(test_indices) == 114, 'Kích thước split không như mong đợi.'

,toan_bo,train,test
diagnosis,,,
B,357,285,72
M,212,170,42
Tong,569,455,114


## 5. Lưu danh sách index và metadata

Metadata chứa SHA-256 của file dữ liệu sạch. Trước khi chạy notebook mô hình, nhóm có thể đối chiếu SHA-256 để bảo đảm file dữ liệu chưa bị thay đổi hoặc sắp xếp lại sau khi split được tạo.

In [5]:
def sha256_file(path: Path) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as file:
        for chunk in iter(lambda: file.read(8192), b''):
            digest.update(chunk)
    return digest.hexdigest()

splits_dir.mkdir(parents=True, exist_ok=True)
pd.DataFrame({'row_index': train_indices}).to_csv(train_indices_path, index=False)
pd.DataFrame({'row_index': test_indices}).to_csv(test_indices_path, index=False)

metadata = {
    'source_file': str(clean_data_path.relative_to(project_root)),
    'source_sha256': sha256_file(clean_data_path),
    'created_at_utc': datetime.now(timezone.utc).isoformat(),
    'total_rows': int(len(clean_data)),
    'total_columns': int(clean_data.shape[1]),
    'target_column': 'diagnosis',
    'label_encoding_for_split': {'B': 0, 'M': 1},
    'test_size': TEST_SIZE,
    'random_state': RANDOM_STATE,
    'stratify': True,
    'train_rows': int(len(train_indices)),
    'test_rows': int(len(test_indices)),
    'class_counts': {
        'full': clean_data['diagnosis'].value_counts().to_dict(),
        'train': train_data['diagnosis'].value_counts().to_dict(),
        'test': test_data['diagnosis'].value_counts().to_dict(),
    },
}

metadata_path.write_text(
    json.dumps(metadata, ensure_ascii=False, indent=2), encoding='utf-8'
)

print(f'Đã lưu: {train_indices_path}')
print(f'Đã lưu: {test_indices_path}')
print(f'Đã lưu: {metadata_path}')

Đã lưu: D:\Project_AIL303m\data\splits\train_indices.csv
Đã lưu: D:\Project_AIL303m\data\splits\test_indices.csv
Đã lưu: D:\Project_AIL303m\data\splits\split_metadata.json


## 6. Cách dùng ở các notebook mô hình

Các notebook SVM, KNN và Decision Tree phải đọc lại `breast_cancer_clean.csv` và hai file index. Không được gọi `train_test_split` lần nữa. Mẫu sử dụng sẽ là:

```python
data = pd.read_csv(clean_data_path)
train_indices = pd.read_csv(train_indices_path)['row_index']
test_indices = pd.read_csv(test_indices_path)['row_index']

train_data = data.iloc[train_indices]
test_data = data.iloc[test_indices]
```

Sau đó, mỗi notebook chỉ tuning mô hình của mình bằng cross-validation trên `train_data`; `test_data` được giữ lại cho đánh giá cuối cùng.

## 7. Kết luận

Sau khi notebook chạy thành công, `data/splits/` là nguồn split duy nhất của dự án. Mọi kết quả so sánh sau này chỉ có ý nghĩa khi ba mô hình cùng dùng các index này.